# Previsión de ventas por país y familia

Demo 6. De `gold` a una previsión de 28 días, en un space `ml.t3.medium`. Sin endpoints.

**Antes de empezar:** `preparar_datos.py` y `generar_clima.py` han dejado los datos en `ml/`. **El clima es inventado** y el histórico no tiene efecto del clima, así que no debería mejorar el modelo.

**Al terminar:** para el space (*Stop space*), porque cuesta dinero mientras está en marcha.

## Parámetros

In [ ]:
import boto3

REGION = "eu-west-1"
CUENTA = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
BUCKET = f"retail-company-datalake-prod-{CUENTA}"  # con otro sufijo_bucket, cámbialo aquí
BASE = f"s3://{BUCKET}/ml"

La imagen SageMaker Distribution ya trae todo lo que usa el notebook; probado con xgboost 2.1.4, pandas 2.3.3 y s3fs.

In [ ]:
import matplotlib.pyplot as plt
import xgboost as xgb

print("xgboost", xgb.__version__)

Funciones de la preparación. Los tests del repositorio ejecutan esta misma celda.

In [ ]:
from datetime import UTC, datetime

import pandas as pd

HORIZONTE = 28
CLIMA = ["temp_media", "lluvia_mm"]
FEATURES = ["dia_semana", "semana_iso", "mes", "temporada",
            "es_festivo", "es_rebajas", "temp_media", "lluvia_mm",
            "lag_28", "lag_364", "media_28", "pais", "familia"]
FEATURES_SIN_CLIMA = [f for f in FEATURES if f not in CLIMA]
CATEGORICAS = ["pais", "familia", "temporada"]
BOOLEANAS = ["es_festivo", "es_rebajas"]


def serie_completa(ventas, cal, clima, dias_futuros=0):
    """Una fila por día, país y familia: 0 unidades sin venta y vacío en los días futuros."""
    cal = cal.assign(fecha=pd.to_datetime(cal["fecha"]))
    fechas = cal.set_index("fecha_sk")["fecha"]
    primera, ultima = fechas[ventas["fecha_sk"].min()], fechas[ventas["fecha_sk"].max()]
    dias = cal[cal["fecha"].between(primera, ultima + pd.Timedelta(days=dias_futuros))]
    rejilla = pd.MultiIndex.from_product(
        [dias["fecha_sk"], sorted(ventas["pais"].unique()), sorted(ventas["familia"].unique())],
        names=["fecha_sk", "pais", "familia"],
    ).to_frame(index=False)
    df = (rejilla.merge(ventas, on=["fecha_sk", "pais", "familia"], how="left")
                 .merge(cal, on="fecha_sk", how="left")
                 .merge(clima, on=["fecha_sk", "pais"], how="left"))
    df["unidades"] = df["unidades"].astype(float)
    df.loc[df["fecha"] <= ultima, "unidades"] = df["unidades"].fillna(0)
    return df.sort_values(["pais", "familia", "fecha"], ignore_index=True)


def anadir_features(df):
    """Retardos por serie. media_28 va desplazada 28 días: no usa nada que no se conozca al prever."""
    df = df.copy()
    por_serie = df.groupby(["pais", "familia"])["unidades"]
    df["lag_28"] = por_serie.shift(28)
    df["lag_364"] = por_serie.shift(364)  # vacío con 120 días: XGBoost acepta NaN
    df["media_28"] = por_serie.transform(lambda s: s.shift(28).rolling(28, min_periods=1).mean())
    for c in BOOLEANAS:
        df[c] = df[c].astype(int)
    for c in CATEGORICAS:
        df[c] = df[c].astype("category")
    return df


def dividir(df):
    """Split temporal: los últimos 28 días validan."""
    corte = df.fecha.max() - pd.Timedelta(days=HORIZONTE)
    return df[df.fecha <= corte], df[df.fecha > corte], corte


def linea_base(train, valid, corte):
    """Media de las 4 semanas anteriores al corte por serie, aplicada a todo el periodo de validación."""
    ultimas = train[train.fecha > corte - pd.Timedelta(days=HORIZONTE)]
    medias = ultimas.groupby(["pais", "familia"], observed=True)["unidades"].mean().rename("base")
    return valid[["pais", "familia"]].join(medias, on=["pais", "familia"])["base"]


def wape(real, prev):
    return abs(real - prev).sum() / real.sum()


def salida_prevision(futuro, prediccion, fecha_calculo):
    """Las columnas de gold.fct_prevision."""
    return pd.DataFrame({
        "fecha_calculo": fecha_calculo,
        "fecha_sk": futuro["fecha_sk"].to_numpy(),
        "pais": futuro["pais"].astype(str).to_numpy(),
        "familia": futuro["familia"].astype(str).to_numpy(),
        "unidades_previstas": pd.Series(prediccion).clip(lower=0).round(2).to_numpy(),
    })

## 1. Carga

Calendario solo por `fecha_sk`; clima por `fecha_sk` y `pais`.

In [ ]:
ventas = pd.read_parquet(f"{BASE}/ventas_diarias/")
cal = pd.read_parquet(f"{BASE}/calendario/")  # gold.dim_fecha
clima = pd.read_parquet(f"{BASE}/clima/")     # sintético
print(f"ventas {ventas.shape}, calendario {cal.shape}, clima {clima.shape}")
print(cal[["temporada", "es_festivo", "es_rebajas"]].dtypes)
ventas.head()

## 2. Serie completa

Un día sin ventas de una familia también es un dato: 0 unidades. Incluye los 28 días futuros, con las unidades vacías.

In [ ]:
todo = serie_completa(ventas, cal, clima, dias_futuros=HORIZONTE)
print(f"{len(todo)} filas, {todo.fecha.nunique()} días")
print(f"{todo.pais.nunique()} países, {todo.familia.nunique()} familias")
todo.tail()

## 3. Features

In [ ]:
todo = anadir_features(todo)
df = todo[todo["unidades"].notna()]
futuro = todo[todo["unidades"].isna()]
df[["fecha", "pais", "familia", "unidades", "lag_28", "media_28"]].tail()

## 4. Split temporal

In [ ]:
train, valid, corte = dividir(df)
print(f"Corte: {corte:%Y-%m-%d}. Entrenamiento {len(train)} filas, validación {len(valid)} filas.")

## 5. Línea base

La media de las últimas 4 semanas. Si el modelo no la gana, no aporta nada.

In [ ]:
base = linea_base(train, valid, corte)
wape_base = wape(valid["unidades"], base)
print(f"WAPE línea base: {wape_base:.3f}")

## 6. Modelo, con y sin clima

In [ ]:
def entrenar(columnas):
    modelo = xgb.XGBRegressor(
        n_estimators=800, max_depth=6, learning_rate=0.05,
        objective="reg:tweedie", tree_method="hist",
        enable_categorical=True, early_stopping_rounds=50)
    modelo.fit(train[columnas], train["unidades"],
               eval_set=[(valid[columnas], valid["unidades"])], verbose=False)
    return modelo


modelo = entrenar(FEATURES)
modelo_sin_clima = entrenar(FEATURES_SIN_CLIMA)
wape_modelo = wape(valid["unidades"], modelo.predict(valid[FEATURES]))
wape_sin_clima = wape(valid["unidades"], modelo_sin_clima.predict(valid[FEATURES_SIN_CLIMA]))

resumen = pd.Series({"Línea base": wape_base, "XGBoost sin clima": wape_sin_clima, "XGBoost con clima": wape_modelo})
print(resumen.round(3).to_string())
print(f"Mejor iteración: {modelo.best_iteration}")

In [ ]:
ax = resumen.plot.barh(color="#4C78A8")
ax.set_xlabel("WAPE (menos es mejor)")
ax.bar_label(ax.containers[0], fmt="%.3f", padding=3)
ax.invert_yaxis()
plt.show()

## 7. Importancia de las features

Ganancia media de cada feature. El calendario debería pesar más que el clima, que es aleatorio. `lag_364` no aparece: está vacía con 120 días.

In [ ]:
importancia = pd.Series(modelo.get_booster().get_score(importance_type="gain")).sort_values()
ax = importancia.plot.barh(color="#4C78A8")
ax.set_xlabel("Ganancia")
plt.show()

## 8. Previsión de 28 días

Se reentrena con todo el histórico y las iteraciones elegidas. Todos los retardos caen en días conocidos.

In [ ]:
assert futuro["lag_28"].notna().all()

final = xgb.XGBRegressor(
    n_estimators=modelo.best_iteration + 1, max_depth=6, learning_rate=0.05,
    objective="reg:tweedie", tree_method="hist", enable_categorical=True)
final.fit(df[FEATURES], df["unidades"], verbose=False)

fecha_calculo = datetime.now(UTC).date().isoformat()
prevision = salida_prevision(futuro, final.predict(futuro[FEATURES]), fecha_calculo)
ruta = f"{BASE}/predicciones/fecha_calculo={fecha_calculo}/prevision.csv"
prevision.to_csv(ruta, index=False)
print(f"{len(prevision)} filas en {ruta}")
print(f"Del {prevision.fecha_sk.min()} al {prevision.fecha_sk.max()}: {prevision.unidades_previstas.sum():.0f} uds.")
prevision.head()

Ahora, en la terminal: `python demo6-prevision-sagemaker/scripts/cargar_prevision.py`.

## 9. (Opcional) Datos para el training job

Solo para la ruta opcional con `lanzar_entrenamiento.py`.

In [ ]:
GENERAR_RUTA_OPCIONAL = False

if GENERAR_RUTA_OPCIONAL:
    train[FEATURES + ["unidades"]].to_csv(f"{BASE}/train/train.csv", index=False)
    valid[FEATURES + ["unidades"]].to_csv(f"{BASE}/valid/valid.csv", index=False)
    futuro[FEATURES].to_csv(f"{BASE}/entrada_bt/futuro.csv", index=False, header=False)
    print("Escritos ml/train/, ml/valid/ y ml/entrada_bt/.")